# Fit foraging models locally and save the results

The remote MLE pipeline (`get_mle_model_fitting` / `get_fitted_latent`) can return
**degenerate latent variables** for some model variants — e.g. an all-zero `q_value`
array for `QLearning_L1F1_CK1_softmax`, which makes the value latents (sumQ / deltaQ /
chosenQ) useless.

This notebook fits the desired foragers **locally** (like `model_comparison/model_camparison_running.ipynb`)
and **saves the fitted latent variables to disk**. Once saved, `behavior_utils.get_fitted_latent`
(and therefore `generate_behavior_summary`) will use these local results **by default**,
falling back to the remote pipeline only when no local fit exists.

Local fits are stored under `behavior_utils.DEFAULT_LOCAL_FIT_DIR`:

```
<LOCAL_FIT_DIR>/<session_core>/<model_alias>.json
```

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np

from nwb_utils import NWBUtils
from model_fitting import fit_forager_latent
from behavior_utils import (
    DEFAULT_LOCAL_FIT_DIR,
    save_local_fitted_latent,
    load_local_fitted_latent,
    get_fitted_latent,
)

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")
print(f"🗂️  Local fit store: {DEFAULT_LOCAL_FIT_DIR}")

In [ ]:
# =============================================================================
# 2. CONFIGURATION
# =============================================================================
# Session to fit. Use the same session_name convention as the behavior summary
# (the local store is keyed by the session 'core', so .nwb / prefixes are fine).
SESSION_NAME = "863973_2026-08-25_14-18-29"

# Models to fit locally. These aliases must exist in
# ForagerCollection().get_all_foragers()['agent_alias'].
MODEL_ALIASES = [
    "QLearning_L1F1_CK1_softmax",
    "QLearning_L2F1_softmax",
]

# Fitting settings (differential evolution).
WORKERS = 4
SEED = 42

# Where to save. None -> behavior_utils.DEFAULT_LOCAL_FIT_DIR.
LOCAL_FIT_DIR = None

# Re-fit even if a local result already exists.
OVERWRITE = False

print(f"Session      : {SESSION_NAME}")
print(f"Models       : {MODEL_ALIASES}")
print(f"Save dir     : {LOCAL_FIT_DIR or DEFAULT_LOCAL_FIT_DIR}")

In [ ]:
# =============================================================================
# 3. LOAD BEHAVIOR NWB
# =============================================================================
# Searches the default behavior / optogenetics NWB folders for this session.
nwb = NWBUtils.read_behavior_nwb(session_name=SESSION_NAME)
assert nwb is not None, (
    f"Could not load behavior NWB for '{SESSION_NAME}'. "
    "Pass nwb_full_path=... to NWBUtils.read_behavior_nwb if it lives elsewhere."
)

df_trials = nwb.trials.to_dataframe()
n_total = len(df_trials)
n_valid = int((df_trials.animal_response != 2).sum())
print(f"Loaded NWB: {n_total} trials ({n_valid} response, {n_total - n_valid} no-response).")

In [ ]:
# =============================================================================
# 4. FIT EACH MODEL LOCALLY AND SAVE
# =============================================================================
saved_paths = {}

for alias in MODEL_ALIASES:
    existing = load_local_fitted_latent(SESSION_NAME, alias, local_dir=LOCAL_FIT_DIR)
    if existing is not None and not OVERWRITE:
        print(f"[SKIP] {alias}: local fit already exists (set OVERWRITE=True to re-fit).")
        continue

    print(f"[FIT ] {alias} ...", flush=True)
    fit_dict = fit_forager_latent(
        nwb,
        alias,
        workers=WORKERS,
        seed=SEED,
    )

    path = save_local_fitted_latent(
        SESSION_NAME,
        alias,
        fit_dict,
        local_dir=LOCAL_FIT_DIR,
    )
    saved_paths[alias] = path
    print(f"[SAVE] {alias} -> {path}")

print("\nDone.")
for alias, path in saved_paths.items():
    print(f"  {alias}: {path}")

In [ ]:
# =============================================================================
# 5. SANITY CHECK — confirm the local q_value is populated (not all-zero)
# =============================================================================
for alias in MODEL_ALIASES:
    local = load_local_fitted_latent(SESSION_NAME, alias, local_dir=LOCAL_FIT_DIR)
    if local is None:
        print(f"{alias}: no local fit found.")
        continue

    flv = local.get("fitted_latent_variables") or {}
    q = np.asarray(flv.get("q_value", []), dtype=float)
    cp = np.asarray(flv.get("choice_prob", []), dtype=float)
    n_nonzero = int(np.count_nonzero(q)) if q.size else 0
    print(
        f"{alias}: q_value shape={q.shape}, nonzero={n_nonzero}/{q.size}; "
        f"choice_prob shape={cp.shape}"
    )
    print(f"   params: {local.get('params')}")

## Using the local fits

`get_fitted_latent(session_name, model_alias)` now returns the **local** fit by default
(`use_local=True`). Because `generate_behavior_summary` calls `get_fitted_latent` for each
remote alias, the behavior summary will automatically pick up these saved results — no
further changes required.

To force the remote pipeline instead, call `get_fitted_latent(..., use_local=False)`.

In [ ]:
# =============================================================================
# 6. VERIFY get_fitted_latent now prefers the local fit
# =============================================================================
alias = MODEL_ALIASES[0]
res = get_fitted_latent(session_name=SESSION_NAME, model_alias=alias)
flv = (res or {}).get("fitted_latent_variables") or {}
q = np.asarray(flv.get("q_value", []), dtype=float)
print(f"{alias}: local q_value nonzero={int(np.count_nonzero(q))}/{q.size}")